In [4]:
!pip install -r requirements.txt -qq

In [ ]:
import torch
from transformers import AutoTokenizer

from constants.model_constants import MODEL_CLASS_MAP
from utils.io_utils import prepare_input, derive_step_rewards

In [2]:
def calculate_reward(model_path, problem, steps):
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    model = MODEL_CLASS_MAP[model_path].from_pretrained(model_path, device_map="auto", torch_dtype=torch.float16)
    model = model.eval()
    input_ids, token_masks = prepare_input(
                            model_path, 
                            problem=problem, 
                            steps=steps, 
                            tokenizer=tokenizer
                        )
    with torch.inference_mode():
        logits = model(input_ids).logits

    rewards = derive_step_rewards(
        model_path, 
        logits, 
        token_masks, 
        tokenizer
    )
    return rewards

### peiyi9979/math-shepherd-mistral-7b-prm 

In [3]:
SHEPHERD_MODEL_PATH = 'peiyi9979/math-shepherd-mistral-7b-prm'
data = {
    "problem": "Janet\u2019s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?",
    "steps": [
        "Step 1: Janet's ducks lay 16 eggs per day.",
        "Step 2: She eats three for breakfast every morning, so she has 16 - 3 = 13 eggs left.",
        "Step 3: She bakes muffins for her friends every day with four eggs, so she has 13 - 4 = 9 eggs left.",
        "Step 4: She sells the remainder at the farmers' market daily for $2 per fresh duck egg, so she makes 9 * $2 = $18 every day at the farmers' market. The answer is: 18"        
    ]
}

In [ ]:
rewards = calculate_reward(SHEPHERD_MODEL_PATH, data["problem"], data["steps"])
print(rewards)
# Rewards should be close to: [[0.9955, 0.9958, 0.9983, 0.9957]]

### Qwen/Qwen2.5-Math-PRM-7B

In [3]:
QWEN_MODEL_PATH = "hf_cache/Qwen--Qwen2.5-Math-PRM-7B"
data = {
    "problem": "Sue lives in a fun neighborhood.  One weekend, the neighbors decided to play a prank on Sue.  On Friday morning, the neighbors placed 18 pink plastic flamingos out on Sue's front yard.  On Saturday morning, the neighbors took back one third of the flamingos, painted them white, and put these newly painted white flamingos back out on Sue's front yard.  Then, on Sunday morning, they added another 18 pink plastic flamingos to the collection. At noon on Sunday, how many more pink plastic flamingos were out than white plastic flamingos?",
    "steps": [
        "To find out how many more pink plastic flamingos were out than white plastic flamingos at noon on Sunday, we can break down the problem into steps. First, on Friday, the neighbors start with 18 pink plastic flamingos.",
        "On Saturday, they take back one third of the flamingos. Since there were 18 flamingos, (1/3 \\times 18 = 6) flamingos are taken back. So, they have (18 - 6 = 12) flamingos left in their possession. Then, they paint these 6 flamingos white and put them back out on Sue's front yard. Now, Sue has the original 12 pink flamingos plus the 6 new white ones. Thus, by the end of Saturday, Sue has (12 + 6 = 18) pink flamingos and 6 white flamingos.",
        "On Sunday, the neighbors add another 18 pink plastic flamingos to Sue's front yard. By the end of Sunday morning, Sue has (18 + 18 = 36) pink flamingos and still 6 white flamingos.",
        "To find the difference, subtract the number of white flamingos from the number of pink flamingos: (36 - 6 = 30). Therefore, at noon on Sunday, there were 30 more pink plastic flamingos out than white plastic flamingos. The answer is (\\boxed{30})."
    ]
}

In [4]:
rewards = calculate_reward(QWEN_MODEL_PATH, data["problem"], data["steps"])
print(rewards)
# Rewards should be close to: [[1.0, 0.1572265625, 0.9765625, 1.0]]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of the model checkpoint at hf_cache/Qwen--Qwen2.5-Math-PRM-7B were not used when initializing Qwen2ForProcessRewardModel: ['lm_head.weight']
- This IS expected if you are initializing Qwen2ForProcessRewardModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Qwen2ForProcessRewardModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


[[0.99951171875, 0.158447265625, 0.97412109375, 0.99951171875]]


### RLHFlow/Llama3.1-8B-PRM-Mistral-Data

In [4]:
RLHF_MODEL_PATH = "RLHFlow/Llama3.1-8B-PRM-Mistral-Data"
data = {
    "problem": "Cindy's math and science books weigh 2 pounds each. Her French book weighs 4 pounds and her English book weighs 3 pounds. Her history book weighs twice as much as her English book. If Cindy carries all of her books at once, what will be the total weight of the books she is carrying?",
    "steps": [
        "To determine the total weight of all Cindy's books, we need to calculate the weight of each book individually and then sum these weights.", 
        "First, for the math and science books:\n- Each math book weighs 2 pounds.\n- Each science book weighs 2 pounds.\n- Cindy has 2 math books and 2 science books.\n- Total weight of math books: \\(2 \\text{ books} \\times 2 \\text{ pounds/book} = 4 \\text{ pounds}\\).\n- Total weight of science books: \\(2 \\text{ books} \\times 2 \\text{ pounds/book} = 4 \\text{ pounds}\\).\n- Combined weight of math and science books: \\(4 \\text{ pounds} + 4 \\text{ pounds} = 8 \\text{ pounds}\\).", "Second, for the French book:\n- The French book weighs 4 pounds.",
        "Third, for the English book:\n- The English book weighs 3 pounds.", "Fourth, for the history book:\n- The history book weighs twice as much as the English book.\n- Weight of the history book: \\(2 \\times 3 \\text{ pounds} = 6 \\text{ pounds}\\).",
        "Finally, for the total weight:\n- Sum of the weights of all the books: \\[ 8 \\text{ pounds} \\text{ (math and science)} + 4 \\text{ pounds} \\text{ (French)} + 3 \\text{ pounds} \\text{ (English)} + 6 \\text{ pounds} \\text{ (history)} = 21 \\text{ pounds} \\]",
        "Therefore, the total weight of the books Cindy is carrying is \\(\\boxed{21}\\) pounds."
    ]
}

In [ ]:
rewards = calculate_reward(RLHF_MODEL_PATH, data["problem"], data["steps"])
print(rewards)
# Rewards should be close to: [[0.83740234375, 0.87060546875, 0.947265625, 0.97412109375, 0.98388671875, 0.986328125, 0.99365234375]]

### RLHFlow/Llama3.1-8B-PRM-Deepseek-Data

In [3]:
RLHF_MODEL_PATH = "RLHFlow/Llama3.1-8B-PRM-Deepseek-Data"
data = {
    "problem": "Cindy's math and science books weigh 2 pounds each. Her French book weighs 4 pounds and her English book weighs 3 pounds. Her history book weighs twice as much as her English book. If Cindy carries all of her books at once, what will be the total weight of the books she is carrying?",
    "steps": [
        "To determine the total weight of all Cindy's books, we need to calculate the weight of each book individually and then sum these weights.", 
        "First, for the math and science books:\n- Each math book weighs 2 pounds.\n- Each science book weighs 2 pounds.\n- Cindy has 2 math books and 2 science books.\n- Total weight of math books: \\(2 \\text{ books} \\times 2 \\text{ pounds/book} = 4 \\text{ pounds}\\).\n- Total weight of science books: \\(2 \\text{ books} \\times 2 \\text{ pounds/book} = 4 \\text{ pounds}\\).\n- Combined weight of math and science books: \\(4 \\text{ pounds} + 4 \\text{ pounds} = 8 \\text{ pounds}\\).", 
        "Second, for the French book:\n- The French book weighs 4 pounds.",
        "Third, for the English book:\n- The English book weighs 3 pounds.", 
        "Fourth, for the history book:\n- The history book weighs twice as much as the English book.\n- Weight of the history book: \\(2 \\times 3 \\text{ pounds} = 6 \\text{ pounds}\\).",
        "Finally, for the total weight:\n- Sum of the weights of all the books: \\[ 8 \\text{ pounds} \\text{ (math and science)} + 4 \\text{ pounds} \\text{ (French)} + 3 \\text{ pounds} \\text{ (English)} + 6 \\text{ pounds} \\text{ (history)} = 21 \\text{ pounds} \\]",
        "Therefore, the total weight of the books Cindy is carrying is \\(\\boxed{21}\\) pounds."
    ]
}

In [ ]:
rewards = calculate_reward(RLHF_MODEL_PATH, data["problem"], data["steps"])
print(rewards)
# Rewards should be close to: [[0.9970703125, 0.97412109375, 0.99462890625, 0.998046875, 0.9990234375, 0.994140625, 0.99853515625]]

### Skywork/Skywork-o1-Open-PRM-Qwen-2.5-1.5B

In [3]:
SKYWORK_MODEL_PATH = "hf_cache/Skywork--Skywork-o1-Open-PRM-Qwen-2.5-1.5B"
data = {
    "problem": "Cindy's math and science books weigh 2 pounds each. Her French book weighs 4 pounds and her English book weighs 3 pounds. Her history book weighs twice as much as her English book. If Cindy carries all of her books at once, what will be the total weight of the books she is carrying?",
    "steps": [
        "To determine the total weight of all Cindy's books, we need to calculate the weight of each book individually and then sum these weights.", 
        "First, for the math and science books:\n- Each math book weighs 2 pounds.\n- Each science book weighs 2 pounds.\n- Cindy has 2 math books and 2 science books.\n- Total weight of math books: \\(2 \\text{ books} \\times 2 \\text{ pounds/book} = 4 \\text{ pounds}\\).\n- Total weight of science books: \\(2 \\text{ books} \\times 2 \\text{ pounds/book} = 4 \\text{ pounds}\\).\n- Combined weight of math and science books: \\(4 \\text{ pounds} + 4 \\text{ pounds} = 8 \\text{ pounds}\\).", 
        "Second, for the French book:\n- The French book weighs 4 pounds.",
        "Third, for the English book:\n- The English book weighs 3 pounds.", 
        "Fourth, for the history book:\n- The history book weighs twice as much as the English book.\n- Weight of the history book: \\(2 \\times 3 \\text{ pounds} = 6 \\text{ pounds}\\).",
        "Finally, for the total weight:\n- Sum of the weights of all the books: \\[ 8 \\text{ pounds} \\text{ (math and science)} + 4 \\text{ pounds} \\text{ (French)} + 3 \\text{ pounds} \\text{ (English)} + 6 \\text{ pounds} \\text{ (history)} = 21 \\text{ pounds} \\]",
        "Therefore, the total weight of the books Cindy is carrying is \\(\\boxed{21}\\) pounds."
    ]
}

In [4]:
tokenizer = AutoTokenizer.from_pretrained(SKYWORK_MODEL_PATH)
    
model = MODEL_CLASS_MAP[SKYWORK_MODEL_PATH].from_pretrained(SKYWORK_MODEL_PATH, device_map="auto", torch_dtype=torch.float16)
model = model.eval()
input_ids, token_masks = prepare_input(
                        SKYWORK_MODEL_PATH, 
                        problem=data["problem"], 
                        steps=data["steps"], 
                        tokenizer=tokenizer
                    )
with torch.inference_mode():
    logits = model(input_ids)[-1]

rewards = derive_step_rewards(
    SKYWORK_MODEL_PATH, 
    logits, 
    token_masks, 
    tokenizer
)
print(rewards)
# Rewards should be close to: [[0.9304260015487671, 0.5120438933372498, 0.6100339889526367, 0.5191707015037537, 0.57666015625, 0.787788450717926, 0.8843944668769836]]

/home/udbhav/prm-inference/model_utils/models/skywork_o1_open_prm/modeling_base.py:264: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = loading_func(filename if n

[[0.9307366013526917, 0.5120867490768433, 0.609718382358551, 0.5187934637069702, 0.5761072039604187, 0.7873608469963074, 0.8840384483337402]]
